In [ ]:

import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

os.environ["OPENAI_API_KEY"]
df = pd.read_csv("bugs.csv")

# Combine title + description
df['text'] = df['title'] + " " + df['description']

# Labels: 1 = Valid Bug, 0 = Invalid
df['label'] = df['label'].map({'bug':1, 'invalid':0})

df = df.dropna(subset=['text', 'label'])  # Drop any rows with missing values

In [ ]:
# Split data
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
print(f"\nTraining samples : {len(train_df)}")
print(f"Test samples     : {len(test_df)}")
# Convert to HuggingFace Dataset
train_dataset = Dataset.from_pandas(train_df[['text', 'label']])
test_dataset  = Dataset.from_pandas(test_df[['text', 'label']])
# Store test labels for evaluation later
test_labels = test_df['label'].tolist()

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

def tokenize(examples):
    return tokenizer(
        examples['text'],
        padding="max_length",
        truncation=True,
        max_length=256
    )

In [ ]:
# Apply tokenization
train_data = train_dataset.map(tokenize, batched=True)
test_data = test_dataset.map(tokenize, batched=True)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2   # Bug (1) and invalid (0)
)
training_args = TrainingArguments(
    output_dir='./results',
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,                # standard for BERT fine-tuning
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
    metric_for_best_model="eval_loss",
    logging_dir='./logs',
    logging_steps=10
)


# Create Trainer & Train
#
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data
)

trainer.save_model("./model")
tokenizer.save_pretrained("./model")
print("Model saved to ./model")


preds = trainer.predict(test_data)

In [ ]:
from sklearn.metrics import accuracy_score

preds = trainer.predict(test_data)
accuracy = accuracy_score(test_labels, preds.predictions.argmax(-1))

print("Accuracy:", accuracy)

In [ ]:
#RAG Implementation
from langchain.vectorstores import FAISS
from langchain.embeddings import OpenAIEmbeddings

embedding = OpenAIEmbeddings()

vector_db = FAISS.from_texts(df['text'].tolist(), embedding)

In [ ]:
from langchain.chat_models import ChatOpenAI

llm = ChatOpenAI()

response = llm.invoke(f"""
Explain why this bug is valid:
Context: {docs}
""")

print(response)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(test_labels, preds.predictions.argmax(-1)))

In [ ]:
# Building an AI Bug Assistant chatbot

pip install streamlit langchain faiss-cpu openai transformers torch

In [ ]:
import streamlit as st
from transformers import pipeline
from langchain.vectorstores import FAISS
from langchain.embeddings import OpenAIEmbeddings
from langchain.chat_models import ChatOpenAI

#Load ML Model
classifier = pipeline("text-classification", model="./model")

# Load Vector DB
embedding = OpenAIEmbeddings()
vector_db = FAISS.load_local("faiss_index", embedding)
vector_db.save_local("faiss_index")  # Add this in notebook
retriever = vector_db.as_retriever(search_kwargs={"k": 3})

# ---- Load LLM ----
llm = ChatOpenAI(temperature=0)

# ---- UI ----
st.title("🤖 AI Bug Assistant")

if "messages" not in st.session_state:
    st.session_state.messages = []

user_input = st.chat_input("Describe your issue...")

if user_input:
    # Save user message
    st.session_state.messages.append({"role": "user", "content": user_input})

    # ---- Step 1: ML Prediction ----
    pred = classifier(user_input)[0]
    label = pred['label']
    confidence = round(pred['score'] * 100, 2)

    # ---- Step 2: RAG Retrieval ----
    docs = retriever.get_relevant_documents(user_input)

    context = " ".join([doc.page_content for doc in docs])

    # ---- Step 3: LLM Explanation ----
    prompt = f"""
    You are a software testing expert.

    Bug: {user_input}
    Prediction: {label}

    Context: {context}

    Explain clearly why this is {label}.
    """

    response = llm.invoke(prompt)

    final_answer = f"""
Prediction: {label} ({confidence}%)

Explanation:
{response.content}
"""

    # Save bot response
    st.session_state.messages.append({"role": "assistant", "content": final_answer})

# ---- Display Chat ----
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])

In [ ]:
#Confidence

if confidence < 70:
    label = "UNCERTAIN"


st.write("### 🔍 Similar Issues Found:")
for doc in docs:
    st.write("-", doc.page_content[:100])


if st.button("👍 Correct"):
    st.success("Thanks for the feedback!")
if st.button("👎 Incorrect"):
    st.warning("We'll use this to improve the model.")

